# Treccani DBI — scale-up run (300 individuals) vs Cultura database

Same validated pipeline as notebook 1 (prompt v6), applied to 300 fresh individuals:

1. Sample 300 individuals with a Dizionario Biografico degli Italiani (Treccani) biography — 60 per floruit bin, regions balanced — excluding the 30 individuals of the two pilot samples. Binning uses `floruit_year` with a midpoint-of-period fallback (only 454 DBI individuals have an explicit `floruit_year`).
2. **Gemini 3.5 Flash, step 1** — extract the most granular floruit location, the floruit period, and word-for-word verbatim evidence (original + English). The floruit is the period of **attested active work** — never the lifespan; active-until-death must be proven; unknown slots stay empty. Three date slots: `start` only when the first activity is attested, `end` only when the last activity is attested, and other attested activity years go to `middle` (an activity date never proves it was the last one). Evidence may span several extracts (`extract 1:`, `extract 2:`, …); each must be an exact substring of the biography and must explicitly contain the location name / the dates. When the text has no precise years, the dating expression as written (e.g. *prima metà del XIII secolo*) is recorded and verified instead. All checks are programmatic.
3. **Gemini 3.5 Flash, step 2** — map the location to the **smallest** Cliopatria polity governing it during the floruit (no supra-level entities); most-likely fit if the exact polity is absent; `None` only when nothing plausibly corresponds.
4. Output a TSV with empty annotation columns (single annotator), then append
   a programmatic (no-AI) comparison against the Cultura database with
   accuracy scores.

In [1]:
import json, os, random, time, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import duckdb
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "humans_clean.duckdb").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DB_PATH = PROJECT_ROOT / "data" / "humans_clean.duckdb"
OUT_DIR = PROJECT_ROOT / "annotations" / "treccani_validation"
CACHE_DIR = OUT_DIR / "_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_PER_BIN = 60                     # 60 individuals x 5 period bins = 300
N_TOTAL = N_PER_BIN * 5
FETCH_WORKERS = 8
LLM_WORKERS = 12
MODEL = "google/gemini-3.5-flash"
PROMPT_VERSION = "v6"              # bump to invalidate the LLM caches
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
BIO_CHAR_CAP = 25_000
PERIOD_BINS = [-10_000, 1300, 1500, 1650, 1800, 3000]
PERIOD_LABELS = ["<1300", "1300-1500", "1500-1650", "1650-1800", "1800+"]

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ["OPEN_ROUTER_API"]
random.seed(SEED)

con = duckdb.connect(str(DB_PATH), read_only=True)
print("db:", DB_PATH.name, "| model:", MODEL)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/lib/python3.

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/lib/python3.

AttributeError: _ARRAY_API not found

db: humans_clean.duckdb | model: google/gemini-3.5-flash


## 1. Candidates — DBI individuals with floruit, Italian birthplace and a Cultura polity

In [2]:
candidates = con.execute("""
    WITH dbi AS (
        SELECT wikidata_id, any_value(value) AS dbi_id
        FROM identifiers WHERE property_id = 'P1986' AND value IS NOT NULL
        GROUP BY wikidata_id
    ),
    cultura AS (
        SELECT wikidata_id, string_agg(DISTINCT polity_name, '; ') AS cultura_polities
        FROM individuals_cliopatria GROUP BY wikidata_id
    )
    SELECT i.wikidata_id, i.name_en, d.dbi_id,
           coalesce(fp.floruit_year,
                    (fp.floruit_period_start + fp.floruit_period_end) // 2) AS floruit_year,
           fp.floruit_period_start, fp.floruit_period_end,
           p.name_en AS birthplace, p.lat AS birth_lat,
           c.cultura_polities
    FROM individuals i
    JOIN dbi d       ON d.wikidata_id = i.wikidata_id
    JOIN individuals_floruit_period fp ON fp.wikidata_id = i.wikidata_id
    JOIN individuals_keys k ON k.wikidata_id = i.wikidata_id
    JOIN places p    ON p.id = k.birthcity_id
                    AND p.iso_a3_code = 'ITA' AND p.lat IS NOT NULL
    JOIN cultura c   ON c.wikidata_id = i.wikidata_id
    WHERE fp.floruit_period_start IS NOT NULL
      AND fp.floruit_period_end IS NOT NULL
      AND i.non_human = 0
      AND i.name_en IS NOT NULL
""").df()

# Stable order so the seeded sample is reproducible across runs.
candidates = candidates.sort_values("wikidata_id").reset_index(drop=True)
print(f"candidates: {len(candidates):,}")

# Exclude the individuals of the two pilot samples so the 300 are all new.
pilot_ids = set()
for fname in ["treccani_validation_sample10.tsv", "treccani_validation_sample20.tsv"]:
    path = OUT_DIR / fname
    if path.exists():
        pilot_ids |= set(pd.read_csv(path, sep="\t")["wikidata_id"])
candidates = candidates[~candidates["wikidata_id"].isin(pilot_ids)].reset_index(drop=True)
print(f"candidates after excluding {len(pilot_ids)} pilot individuals: {len(candidates):,}")
candidates.head()

candidates: 22,519
candidates after excluding 30 pilot individuals: 22,489


,wikidata_id,name_en,dbi_id,floruit_year,floruit_period_start,floruit_period_end,birthplace,birth_lat,cultura_polities
0,Q100076969,Luigi Caracciolo,luigi-caracciolo,1881,1876,1887,Andria,41.231667,Kingdom of Great Britain; British Empire
1,Q100077086,Ulisse Corticelli,ulisse-corticelli,1863,1850,1876,Ravenna,44.416111,Papal States; Kingdom of Italy; Kingdom of Sar...
2,Q100077443,Enrico Contessa,enrico-contessa,1919,1906,1932,Turin,45.079167,Kingdom of Italy
3,Q100077655,Andrea D'Angeli,andrea-d-angeli,1910,1897,1923,Padua,45.407778,Kingdom of Italy
4,Q100137528,Francesco Malgeri,francesco-malgeri,1942,1929,1955,Messina,38.193611,Republic of Italy; Kingdom of Italy


## 2. Stratified sample — region of origin × floruit period

In [3]:
# Region of origin from birthplace latitude (Italian macro-areas).
def region_of(lat):
    if lat >= 44.0:
        return "North"
    if lat >= 41.5:
        return "Center"
    return "South & Islands"

candidates["region"] = candidates["birth_lat"].apply(region_of)
candidates["period_bin"] = pd.cut(
    candidates["floruit_year"], bins=PERIOD_BINS, labels=PERIOD_LABELS)

print(candidates.groupby(["period_bin", "region"], observed=True).size().unstack(fill_value=0))

# N_PER_BIN individuals per period bin, regions balanced (up to a third each).
per_region = -(-N_PER_BIN // 3)
rows = []
for label in PERIOD_LABELS:
    pool = candidates[candidates["period_bin"] == label]
    pool = pool.sample(frac=1, random_state=SEED)
    picked = pool.groupby("region", group_keys=False).head(per_region).head(N_PER_BIN)
    rest = pool.drop(picked.index)
    picked = pd.concat([picked, rest]).head(N_PER_BIN)
    rows.append(picked)

sample = pd.concat(rows).reset_index(drop=True)
assert len(sample) == N_TOTAL and sample["wikidata_id"].is_unique
sample["dbi_url"] = ("https://www.treccani.it/enciclopedia/"
                     + sample["dbi_id"] + "_(Dizionario-Biografico)/")
sample[["wikidata_id", "name_en", "region", "period_bin"]]

region      Center  North  South & Islands
period_bin                                
<1300          433    413              163
1300-1500      987   1233              162
1500-1650     1525   2326              403
1650-1800     1149   2177              501
1800+         3026   5871             2120


,wikidata_id,name_en,region,period_bin
0,Q3762081,Giacomo Dondulo,North,<1300
1,Q3765155,Giordano Forzatè,North,<1300
2,Q39454,Arator,North,<1300
3,Q349351,Richard of San Germano,South & Islands,<1300
4,Q26246988,Nicola Paglia,South & Islands,<1300
...,...,...,...,...
295,Q26222255,Federico Hermanin,South & Islands,1800+
296,Q3737934,Fabrizia Ramondino,South & Islands,1800+
297,Q63290079,Guglielmo Capitelli,South & Islands,1800+
298,Q3768934,Girolamo Giusso,South & Islands,1800+


## 3. Fetch the DBI biographies

The article body is embedded in the page's `__NEXT_DATA__` JSON payload.

In [4]:
PAGES_PATH = CACHE_DIR / "dbi_pages.jsonl"
UA = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}


def fetch_dbi(row):
    r = requests.get(row["dbi_url"], headers=UA, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    nd = soup.find("script", id="__NEXT_DATA__")
    content = json.loads(nd.string)["props"]["pageProps"]["data"]["content"]
    text = BeautifulSoup(content, "html.parser").get_text(" ", strip=True)
    return {"wikidata_id": row["wikidata_id"], "dbi_url": row["dbi_url"], "text": text}


pages = {}
if PAGES_PATH.exists():
    for line in PAGES_PATH.open():
        r = json.loads(line)
        pages[r["wikidata_id"]] = r

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in pages]
if todo:
    with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as ex, PAGES_PATH.open("a") as f:
        futs = [ex.submit(fetch_dbi, row) for row in todo]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="fetch DBI"):
            r = fut.result()
            pages[r["wikidata_id"]] = r
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

assert all(qid in pages and len(pages[qid]["text"]) > 200 for qid in sample["wikidata_id"])
pd.DataFrame([{"wikidata_id": q, "chars": len(pages[q]["text"])} for q in sample["wikidata_id"]])

,wikidata_id,chars
0,Q3762081,17492
1,Q3765155,13858
2,Q39454,21001
3,Q349351,20863
4,Q26246988,13095
...,...,...
295,Q26222255,23102
296,Q3737934,13917
297,Q63290079,23486
298,Q3768934,13995


## 4. LLM step 1 — floruit location, floruit period, verbatim evidence

In [5]:
_tls = threading.local()


def _session():
    if not hasattr(_tls, "s"):
        s = requests.Session()
        s.headers.update({"Content-Type": "application/json",
                          "Authorization": f"Bearer {API_KEY}",
                          "HTTP-Referer": "https://bunka.ai/",
                          "X-Title": "Cultura Treccani validation"})
        _tls.s = s
    return _tls.s


def parse_json_loose(content):
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass
    return json.loads(content[content.find("{"): content.rfind("}") + 1],
                      strict=False)


def call_gemini(system, user, max_retries=6):
    last_err = ""
    for attempt in range(max_retries):
        # temperature 0 is deterministic: after 2 malformed-JSON answers,
        # nudge the sampling and remind the model to escape quotes.
        msg = user
        if attempt >= 2:
            msg += ("\n\nIMPORTANT: your previous answer was INVALID JSON. "
                    "Escape all double quotes inside string values. "
                    "Output one valid JSON object only.")
        body = {"model": MODEL,
                "messages": [{"role": "system", "content": system},
                             {"role": "user", "content": msg}],
                "temperature": 0 if attempt < 2 else 0.4,
                "response_format": {"type": "json_object"}}
        try:
            r = _session().post(OPENROUTER_URL, json=body, timeout=180)
            if r.status_code in (408, 429, 500, 502, 503, 504):
                last_err = f"http_{r.status_code}"
                time.sleep(1.5 * 2 ** attempt)
                continue
            r.raise_for_status()
            return parse_json_loose(r.json()["choices"][0]["message"]["content"])
        except (requests.RequestException, ValueError, KeyError) as e:
            last_err = f"{type(e).__name__}: {e}"
            time.sleep(1.5 * 2 ** attempt)
    raise RuntimeError(last_err)


STEP1_SYSTEM = """You are an expert historian reading a biography from the \
Dizionario Biografico degli Italiani (Treccani), written in Italian.

Your task: identify the ONE most granular location (a city, a region, or \
similar) associated with the individual's FLORUIT — the period during which \
they made their principal contribution — and the floruit period itself.

An individual may be linked to several locations during their active years: \
you must pick the single location best supported by the text as the place of \
their principal activity (not necessarily birthplace or deathplace).

Return STRICT JSON with exactly these keys:
  floruit_location          : most granular location name (English exonym if one exists, e.g. "Florence", "Rome")
  floruit_location_original : the location name EXACTLY as it is written in the text (e.g. "Firenze")
  floruit_location_type     : "city" | "region" | "other"
  floruit_start             : integer year (negative = BCE), or null
  floruit_middle            : ARRAY of integer years (possibly empty)
  floruit_end               : integer year, or null
  floruit_dates_precise     : true if the text gives explicit years; false if
                              the period is expressed vaguely (e.g. a century)
  floruit_period_as_written : the dating expression EXACTLY as written in the
                              text (e.g. "prima metà del XIII secolo", "1232-1239")
  floruit_period_en         : English rendering (e.g. "first half of the 13th century")
  evidence_location_verbatim : ARRAY of verbatim extracts (original language) supporting the location
  evidence_location_en       : ARRAY of English translations, same order
  evidence_floruit_verbatim  : ARRAY of verbatim extracts supporting the floruit period
  evidence_floruit_en        : ARRAY of English translations, same order
  reasoning                 : 1-3 sentences in English

FLORUIT RULES — the floruit is the period of ACTIVE WORK, not the lifespan.
Fill each of the three slots ONLY when you know it:
- floruit_start : ONLY if the text attests the FIRST time the person was
  active — the first time they are linked to a work they produced or to some
  impact (a debut, "starting from...", a first appointment). Else null.
- floruit_end   : ONLY if the text attests the LAST time the person was
  active — an explicitly final activity ("held the role until...", "l'ultima
  notizia...", a last documented work presented as such, retirement or
  documented cessation). The finality MUST be visible in one of your quoted
  floruit extracts — if none of your extracts explicitly marks the activity
  as the last one, put the year in floruit_middle instead. Else null.
- floruit_middle: attested activity years that are NEITHER a proven first nor
  a proven last activity. If you find one or two random activity dates, put
  them here. Activity in a year never proves it was the last one: the person
  may still have been active afterwards.
- Birth or death dates are NOT floruit evidence. Someone may well be active
  until death, but that must be PROVEN by attested activity at or near the
  death year — never assumed.
- An APPOINTMENT attests activity AT that date; the person may continue to
  have impact afterwards, but that is not yet known.
- If you do not know, do not add anything — null / empty is always better
  than a guess.

Examples:
- "he was engaged in London starting from the autumn of 1731" + "roles
  written for him by Handel in Faramondo, Alessandro Severo and Xerses in
  1738" -> floruit_start = 1731 ("starting from" attests the beginning),
  floruit_middle = [1738] (he may still have been active after 1738),
  floruit_end = null.
- "he graduated in Pisa, in 1899 ... After two years he was appointed
  director of the traveling chair of agriculture" + "He maintained the
  direction of the Station ... for another period of two years until
  December 1950" -> floruit_start = 1901, floruit_end = 1950 (the text
  clearly attests both the first activity and the final one),
  floruit_middle = [].

STRICT VERBATIM RULES — the extracts are the annotator's only evidence:
- Each extract MUST be copied WORD BY WORD from the text: an exact contiguous
  substring, identical spelling, accents and punctuation. No paraphrase, no
  summary, no ellipsis, no stitching of distant sentences into one extract.
- You MAY give SEVERAL extracts from different parts of the text (usually 1-3)
  when the evidence is spread out; each array element is one extract.
- At least one location extract MUST contain the location name as written in
  the text (floruit_location_original). A reader must recognise the location
  from the extracts alone.
- If floruit_dates_precise is true, at least one floruit extract MUST contain
  the year(s) on which floruit_start / floruit_end are based.
- If floruit_dates_precise is false, at least one floruit extract MUST contain
  floruit_period_as_written, and floruit_start / floruit_end are your numeric
  interpretation of it (e.g. "prima metà del XIII secolo" -> 1201-1250).
- The same extract may appear in both evidence arrays.
- Reasoning, floruit_location and floruit_period_en in English; verbatim
  extracts stay in the original language; the *_en arrays are translations."""


def step1_user(row):
    text = pages[row["wikidata_id"]]["text"][:BIO_CHAR_CAP]
    return (f"Individual: {row['name_en']} ({row['wikidata_id']})\n\n"
            f"--- DBI BIOGRAPHY ---\n{text}\n--- END ---\n\n"
            "Extract the requested fields. JSON only.")


STEP1_PATH = CACHE_DIR / f"step1_floruit_location_{PROMPT_VERSION}.jsonl"
step1 = {}
if STEP1_PATH.exists():
    for line in STEP1_PATH.open():
        r = json.loads(line)
        step1[r["wikidata_id"]] = r["extraction"]

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in step1]
if todo:
    with ThreadPoolExecutor(max_workers=LLM_WORKERS) as ex, STEP1_PATH.open("a") as f:
        futs = {ex.submit(call_gemini, STEP1_SYSTEM, step1_user(row)): row["wikidata_id"]
                for row in todo}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="LLM step 1"):
            qid = futs[fut]
            step1[qid] = fut.result()
            f.write(json.dumps({"wikidata_id": qid, "extraction": step1[qid]},
                               ensure_ascii=False) + "\n")

assert len(step1) >= len(sample)
pd.DataFrame([{"wikidata_id": q, **step1[q]} for q in sample["wikidata_id"]])[
    ["wikidata_id", "floruit_location", "floruit_location_original",
     "floruit_location_type", "floruit_start", "floruit_middle", "floruit_end",
     "floruit_dates_precise", "floruit_period_en"]]

LLM step 1:   0%|          | 0/30 [00:00<?, ?it/s]

,wikidata_id,floruit_location,floruit_location_original,floruit_location_type,floruit_start,floruit_middle,floruit_end,floruit_dates_precise,floruit_period_en
0,Q3762081,Venice,Venezia,city,1257.0,"[1266, 1281, 1283]",1288.0,True,1257-1288
1,Q3765155,Padua,Padova,city,1203.0,"[1213, 1224, 1229, 1235, 1237, 1239]",NaN,True,1203-1239
2,Q39454,Rome,Roma,city,NaN,[],544.0,True,544
3,Q349351,San Germano,San Germano,city,1186.0,"[1215, 1222, 1239, 1242]",1243.0,True,1186-1243
4,Q26246988,Rome,Roma,city,1229.0,"[1231, 1233, 1234]",1255.0,True,1229-1255
...,...,...,...,...,...,...,...,...,...
295,Q26222255,Rome,Roma,city,1898.0,"[1900, 1913]",1945.0,True,1898-1945
296,Q3737934,Naples,Napoli,city,1968.0,[1981],2008.0,True,1968-2008
297,Q63290079,Naples,Napoli,city,1868.0,[],1870.0,True,1868-1870
298,Q3768934,Naples,Napoli,city,1864.0,"[1878, 1901]",1913.0,True,1864-1913


### 4b. Verify the verbatims are word-for-word substrings

Whitespace-normalised checks: each quote must be an exact substring of the
biography, the location quote must contain the location name as written in
the text, and the floruit quote must contain the extracted years.

In [6]:
import re


def norm(s):
    # quote characters vary between the page and the model's JSON output —
    # drop them before the word-by-word substring comparison
    s = re.sub(r"[\"'\u201c\u201d\u00ab\u00bb\u2018\u2019\u201a\u2039\u203a]", "", str(s))
    return re.sub(r"\s+", " ", s).strip().lower()


def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def floruit_years(e):
    ys = [e["floruit_start"], e["floruit_end"]] + as_list(e.get("floruit_middle"))
    return [int(y) for y in ys if y is not None]


def verify_step1(qid):
    text, e = norm(pages[qid]["text"]), step1[qid]
    loc_ex = [norm(x) for x in as_list(e["evidence_location_verbatim"])]
    flo_ex = [norm(x) for x in as_list(e["evidence_floruit_verbatim"])]
    years = [str(abs(y)) for y in floruit_years(e)]
    if not years and not e.get("floruit_period_as_written"):
        dates_ok = True                       # nothing claimed, nothing to verify
    elif e.get("floruit_dates_precise"):
        dates_ok = any(y in q for q in flo_ex for y in years)
    else:  # vague dating: the expression as written must be quoted instead
        dates_ok = any(norm(e["floruit_period_as_written"]) in q for q in flo_ex)
    return {
        "wikidata_id": qid,
        "location_quotes_in_text": bool(loc_ex) and all(q in text for q in loc_ex),
        "floruit_quotes_in_text": bool(flo_ex) and all(q in text for q in flo_ex),
        "location_name_in_quote": any(norm(e["floruit_location_original"]) in q
                                      for q in loc_ex),
        "dates_in_quote": dates_ok,
    }


CHECK_MSGS = {
    "location_quotes_in_text": "at least one location extract is NOT an exact word-by-word substring of the biography",
    "floruit_quotes_in_text": "at least one floruit extract is NOT an exact word-by-word substring of the biography",
    "location_name_in_quote": "no location extract contains floruit_location_original",
    "dates_in_quote": "no floruit extract contains the floruit years (or floruit_period_as_written if dates are vague)",
}


def failed_checks(qid):
    v = verify_step1(qid)
    return [CHECK_MSGS[k] for k, ok in v.items() if k in CHECK_MSGS and not ok]


# Corrective retries (up to 3 rounds) for extractions that fail verification.
for round_ in range(3):
    retry = [q for q in sample["wikidata_id"] if failed_checks(q)]
    if not retry:
        break
    with STEP1_PATH.open("a") as f:
        for qid in tqdm(retry, desc=f"step 1 retry (round {round_ + 1})"):
            row = sample.loc[sample["wikidata_id"] == qid].iloc[0]
            feedback = ("\n\nYour previous answer failed these checks:\n- "
                        + "\n- ".join(failed_checks(qid))
                        + "\nEach extract must be ONE contiguous word-by-word "
                          "passage — never join two sentences that are not "
                          "adjacent in the text; use separate extracts instead. "
                          "Copy the quotes WORD BY WORD from the text and retry.")
            step1[qid] = call_gemini(STEP1_SYSTEM, step1_user(row) + feedback)
            f.write(json.dumps({"wikidata_id": qid, "extraction": step1[qid]},
                               ensure_ascii=False) + "\n")

verif = pd.DataFrame([verify_step1(q) for q in sample["wikidata_id"]])
verif["evidence_verified"] = verif.drop(columns="wikidata_id").all(axis=1)
print(f"fully verified: {int(verif['evidence_verified'].sum())}/{len(verif)}")
verif

step 1 retry (round 1):   0%|          | 0/40 [00:00<?, ?it/s]

step 1 retry (round 2):   0%|          | 0/4 [00:00<?, ?it/s]

step 1 retry (round 3):   0%|          | 0/3 [00:00<?, ?it/s]

fully verified: 298/300


,wikidata_id,location_quotes_in_text,floruit_quotes_in_text,location_name_in_quote,dates_in_quote,evidence_verified
0,Q3762081,True,True,True,True,True
1,Q3765155,True,True,True,True,True
2,Q39454,True,True,True,True,True
3,Q349351,True,True,True,True,True
4,Q26246988,True,True,True,True,True
...,...,...,...,...,...,...
295,Q26222255,True,True,True,True,True
296,Q3737934,True,True,True,True,True
297,Q63290079,True,True,True,True,True
298,Q3768934,True,True,True,True,True


## 5. LLM step 2 — map the location to a Cliopatria polity

For each individual the model chooses from the Cliopatria polities whose
period overlaps the extracted floruit.

In [7]:
def cliopatria_candidates(start, end):
    return con.execute("""
        SELECT polity_id, polity_name,
               min(from_year) AS from_year, max(to_year) AS to_year
        FROM polities_periods_cliopatria
        WHERE to_year >= ? AND from_year <= ?
        GROUP BY polity_id, polity_name
        ORDER BY polity_name
    """, [start, end]).df()


STEP2_SYSTEM = """You are an expert historical geographer.

Given a location, a floruit period, and the list of Cliopatria polities \
active during that period, choose the ONE polity that governed the location \
during the floruit period.

Return STRICT JSON with exactly these keys:
  polity_id    : integer id copied from the candidate list, or null
  polity_name  : name copied verbatim from the candidate list, or "None"
  confidence   : "high" | "medium" | "low"
  reasoning    : 1-3 sentences in English

Rules:
- polity_id and polity_name MUST come from the candidate list, no invention.
- Choose the SMALLEST (most local, most granular) polity that governed the
  location — a city-state, duchy, county or kingdom — NOT a supra-level
  entity that merely contains it (empire, personal union, alliance,
  "Minor States" aggregate, allegiance/vassalage constructs), unless no
  lower-level candidate covers the location.
- If sovereignty changed during the period, choose the polity covering the
  largest share of the period.
- If the exact governing polity is not in the list, choose the candidate
  MOST LIKELY to fit the location (geographically and politically closest),
  and lower the confidence accordingly.
- ONLY if no candidate could plausibly correspond to the location, answer
  polity_id = null and polity_name = "None", and explain in reasoning."""


def step2_user(row, ext):
    ys = floruit_years(ext)
    s, e = min(ys), max(ys)
    cand = cliopatria_candidates(s, e)
    lines = "\n".join(f"id={r.polity_id} | {r.polity_name} | {r.from_year} to {r.to_year}"
                      for r in cand.itertuples())
    return (f"Location: {ext['floruit_location']}\n"
            f"Floruit period: {ext['floruit_start']} to {ext['floruit_end']}\n"
            f"Individual (context only): {row['name_en']}\n\n"
            f"--- CLIOPATRIA CANDIDATE POLITIES ({len(cand)}) ---\n{lines}\n--- END ---\n\n"
            "Choose the polity. JSON only.")


STEP2_PATH = CACHE_DIR / f"step2_polity_mapping_{PROMPT_VERSION}.jsonl"


def step1_key(qid):
    e = step1[qid]
    return [e["floruit_location"], e["floruit_start"],
            as_list(e.get("floruit_middle")), e["floruit_end"]]


# Cache entries are only valid for the step-1 extraction they were based on.
step2 = {}
if STEP2_PATH.exists():
    for line in STEP2_PATH.open():
        r = json.loads(line)
        if r["wikidata_id"] in step1 and r.get("key") == step1_key(r["wikidata_id"]):
            step2[r["wikidata_id"]] = r["mapping"]

# No extracted floruit at all -> no candidate window, polity is None by construction.
for qid in sample["wikidata_id"]:
    if qid not in step2 and not floruit_years(step1[qid]):
        step2[qid] = {"polity_id": None, "polity_name": "None", "confidence": "low",
                      "reasoning": "No floruit period could be extracted from the text."}

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in step2]
if todo:
    with ThreadPoolExecutor(max_workers=LLM_WORKERS) as ex, STEP2_PATH.open("a") as f:
        futs = {ex.submit(call_gemini, STEP2_SYSTEM,
                          step2_user(row, step1[row["wikidata_id"]])): row["wikidata_id"]
                for row in todo}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="LLM step 2"):
            qid = futs[fut]
            step2[qid] = fut.result()
            f.write(json.dumps({"wikidata_id": qid, "key": step1_key(qid),
                                "mapping": step2[qid]},
                               ensure_ascii=False) + "\n")

assert len(step2) >= len(sample)
pd.DataFrame([{"wikidata_id": q, **step2[q]} for q in sample["wikidata_id"]])[
    ["wikidata_id", "polity_id", "polity_name", "confidence"]]

LLM step 2:   0%|          | 0/300 [00:00<?, ?it/s]

,wikidata_id,polity_id,polity_name,confidence
0,Q3762081,397.0,Republic of Venice,high
1,Q3765155,523.0,Holy Roman Empire,medium
2,Q39454,269.0,Eastern Roman Empire,high
3,Q349351,680.0,Kingdom of Sicily,high
4,Q26246988,419.0,Papal States,high
...,...,...,...,...
295,Q26222255,350.0,Kingdom of Italy,high
296,Q3737934,1428.0,Republic of Italy,high
297,Q63290079,350.0,Kingdom of Italy,high
298,Q3768934,350.0,Kingdom of Italy,high


## 6. Labeling TSV

One row per individual. `annot_*` columns are empty — to be filled by the
annotator (yes / no / notes). This set validates the LLM extraction and
mapping only — no Cultura comparison columns.

In [8]:
def fmt_extracts(x):
    return " | ".join(f"extract {i + 1}: {t}" for i, t in enumerate(as_list(x)))


records = []
for _, row in sample.iterrows():
    qid = row["wikidata_id"]
    e1, e2 = step1[qid], step2[qid]
    records.append({
        "wikidata_id": qid,
        "name": row["name_en"],
        "dbi_url": row["dbi_url"],
        "region": row["region"],
        "period_bin": str(row["period_bin"]),
        "llm_floruit_location": e1["floruit_location"],
        "llm_floruit_location_original": e1["floruit_location_original"],
        "llm_floruit_location_type": e1["floruit_location_type"],
        "llm_floruit_start": e1["floruit_start"],
        "llm_floruit_middle": "; ".join(str(y) for y in as_list(e1.get("floruit_middle"))),
        "llm_floruit_end": e1["floruit_end"],
        "llm_floruit_dates_precise": e1["floruit_dates_precise"],
        "llm_floruit_period_as_written": e1["floruit_period_as_written"],
        "llm_floruit_period_en": e1["floruit_period_en"],
        "evidence_location_verbatim": fmt_extracts(e1["evidence_location_verbatim"]),
        "evidence_location_en": fmt_extracts(e1["evidence_location_en"]),
        "evidence_floruit_verbatim": fmt_extracts(e1["evidence_floruit_verbatim"]),
        "evidence_floruit_en": fmt_extracts(e1["evidence_floruit_en"]),
        "llm_step1_reasoning": e1["reasoning"],
        "llm_polity_id": e2["polity_id"],
        "llm_polity_name": e2["polity_name"],
        "llm_polity_confidence": e2["confidence"],
        "llm_step2_reasoning": e2["reasoning"],
        "annot_location_ok": "",
        "annot_floruit_ok": "",
        "annot_polity_ok": "",
        "annot_notes": "",
    })

out = pd.DataFrame(records)
out = out.merge(verif, on="wikidata_id")
assert out["wikidata_id"].is_unique and len(out) == N_TOTAL

OUT_PATH = OUT_DIR / "treccani_validation_sample300.tsv"
out.to_csv(OUT_PATH, sep="\t", index=False)
print("saved:", OUT_PATH)
out[["name", "region", "period_bin", "llm_floruit_location",
     "llm_floruit_start", "llm_floruit_middle", "llm_floruit_end",
     "llm_floruit_period_en", "llm_polity_name", "evidence_verified"]]

saved: /Users/charlesdedampierre/Desktop/Rsearch Folder/cultura/cultura_database/annotations/treccani_validation/treccani_validation_sample300.tsv


,name,region,period_bin,llm_floruit_location,llm_floruit_start,llm_floruit_middle,llm_floruit_end,llm_floruit_period_en,llm_polity_name,evidence_verified
0,Giacomo Dondulo,North,<1300,Venice,1257.0,1266; 1281; 1283,1288.0,1257-1288,Republic of Venice,True
1,Giordano Forzatè,North,<1300,Padua,1203.0,1213; 1224; 1229; 1235; 1237; 1239,NaN,1203-1239,Holy Roman Empire,True
2,Arator,North,<1300,Rome,NaN,,544.0,544,Eastern Roman Empire,True
3,Richard of San Germano,South & Islands,<1300,San Germano,1186.0,1215; 1222; 1239; 1242,1243.0,1186-1243,Kingdom of Sicily,True
4,Nicola Paglia,South & Islands,<1300,Rome,1229.0,1231; 1233; 1234,1255.0,1229-1255,Papal States,True
...,...,...,...,...,...,...,...,...,...,...
295,Federico Hermanin,South & Islands,1800+,Rome,1898.0,1900; 1913,1945.0,1898-1945,Kingdom of Italy,True
296,Fabrizia Ramondino,South & Islands,1800+,Naples,1968.0,1981,2008.0,1968-2008,Republic of Italy,True
297,Guglielmo Capitelli,South & Islands,1800+,Naples,1868.0,,1870.0,1868-1870,Kingdom of Italy,True
298,Girolamo Giusso,South & Islands,1800+,Naples,1864.0,1878; 1901,1913.0,1864-1913,Kingdom of Italy,True


## 7. Comparison with the Cultura database (pure functions, no AI)

- **Polity**: validated if the LLM polity is among the polities assigned to the
  individual in `individuals_cliopatria` (match on polity id, name as fallback).
- **Floruit**: the LLM interval is [min, max] of the attested slot years
  (a single middle date gives a point interval). Validated if it lies inside
  the Cultura floruit, or if the two overlap on at least 50% of the LLM
  interval.

In [9]:
qids = list(sample["wikidata_id"])
db = con.execute("""
    SELECT wikidata_id,
           list(DISTINCT polity_id)                 AS cultura_polity_ids,
           string_agg(DISTINCT polity_name, '; ')   AS cultura_polities
    FROM individuals_cliopatria
    WHERE wikidata_id IN (SELECT unnest(?::VARCHAR[]))
    GROUP BY wikidata_id
""", [qids]).df().set_index("wikidata_id")


def polity_validated(qid):
    e2 = step2[qid]
    if qid not in db.index or e2["polity_id"] is None:
        return False
    row = db.loc[qid]
    if int(e2["polity_id"]) in [int(i) for i in row["cultura_polity_ids"]]:
        return True
    names = [n.strip().lower() for n in row["cultura_polities"].split(";")]
    return str(e2["polity_name"]).strip().lower() in names


def floruit_validated(qid, db_start, db_end):
    ys = floruit_years(step1[qid])
    if not ys or pd.isna(db_start) or pd.isna(db_end):
        return None
    ls, le = min(ys), max(ys)
    inside = ls >= db_start and le <= db_end
    overlap = max(0, min(le, db_end) - max(ls, db_start) + 1)
    return bool(inside or overlap >= 0.5 * (le - ls + 1))


cmp_rows = []
for _, row in sample.iterrows():
    qid = row["wikidata_id"]
    ys = floruit_years(step1[qid])
    cmp_rows.append({
        "wikidata_id": qid,
        "cultura_polities": db.loc[qid, "cultura_polities"] if qid in db.index else "",
        "auto_polity_validated": polity_validated(qid),
        "cultura_floruit_start": row["floruit_period_start"],
        "cultura_floruit_end": row["floruit_period_end"],
        "llm_floruit_min": min(ys) if ys else None,
        "llm_floruit_max": max(ys) if ys else None,
        "auto_floruit_validated": floruit_validated(
            qid, row["floruit_period_start"], row["floruit_period_end"]),
    })

cmp = pd.DataFrame(cmp_rows)
final = out.merge(cmp, on="wikidata_id")
assert len(final) == N_TOTAL
final.to_csv(OUT_PATH, sep="\t", index=False)
print("saved:", OUT_PATH)

acc_pol = final["auto_polity_validated"].mean()
acc_flo = final["auto_floruit_validated"].mean()
print(f"polity accuracy : {final['auto_polity_validated'].sum()}/{len(final)} = {acc_pol:.0%}")
print(f"floruit accuracy: {final['auto_floruit_validated'].sum()}/{len(final)} = {acc_flo:.0%}")
final[["name", "llm_polity_name", "cultura_polities", "auto_polity_validated",
       "llm_floruit_min", "llm_floruit_max", "cultura_floruit_start",
       "cultura_floruit_end", "auto_floruit_validated"]]

saved: /Users/charlesdedampierre/Desktop/Rsearch Folder/cultura/cultura_database/annotations/treccani_validation/treccani_validation_sample300.tsv
polity accuracy : 231/300 = 77%
floruit accuracy: 248/300 = 83%


,name,llm_polity_name,cultura_polities,auto_polity_validated,llm_floruit_min,llm_floruit_max,cultura_floruit_start,cultura_floruit_end,auto_floruit_validated
0,Giacomo Dondulo,Republic of Venice,Republic of Venice,True,1257,1288,1201,1300,True
1,Giordano Forzatè,Holy Roman Empire,Vassalage of Kingdom of Bohemia to Holy Roman ...,True,1203,1239,1192,1225,True
2,Arator,Eastern Roman Empire,Ostrogothic Kingdom; Allegiance of Ostrogothic...,True,544,544,510,542,False
3,Richard of San Germano,Kingdom of Sicily,Papal States,False,1186,1243,1200,1232,True
4,Nicola Paglia,Papal States,Personal union of Holy Roman Empire with Kingd...,False,1229,1255,1231,1256,True
...,...,...,...,...,...,...,...,...,...
295,Federico Hermanin,Kingdom of Italy,Kingdom of Italy,True,1898,1945,1898,1930,True
296,Fabrizia Ramondino,Republic of Italy,Republic of Italy,True,1968,2008,1992,1997,False
297,Guglielmo Capitelli,Kingdom of Italy,Kingdom of Italy,True,1868,1870,1869,1895,True
298,Girolamo Giusso,Kingdom of Italy,Kingdom of Italy,True,1864,1913,1877,1910,True
